# ShopDesk, Module 3 Section 2 Lab 1: Slash Commands and Skills

A beginner-friendly notebook on turning repeated ShopDesk workflows into **shared tools**: a
project **slash command** in `.claude/commands/`, and a reusable **skill** in `.claude/skills/`
with a `SKILL.md`. We configure `context: fork` to run the skill in an **isolated** subagent and
`allowed-tools` to **restrict** what it can do. Pure-Python cells make the mechanics concrete
offline; a live **Claude Agent SDK** run invokes the command. Runs **Sonnet**
(`claude-sonnet-4-6`) through your **Anthropic API key**.

## The real-world scenario

Every ShopDesk PR gets the same review checklist typed out by hand, and every refund change gets a
manual audit. Both are repeatable, so both belong in the repo as tools: a `/review` **command** any
teammate gets on clone, and a `refund-audit` **skill** that runs in its own context and can only
read, never write.

The question this lab answers: **how do you package a workflow as a command or skill, and how do
`context: fork` and `allowed-tools` keep it isolated and safe?**

## Objectives

- Create a project **slash command** in `.claude/commands/` with frontmatter and `$ARGUMENTS`.
- Build a reusable **skill** (`.claude/skills/<name>/SKILL.md`) with YAML frontmatter.
- Use `context: fork` to isolate execution and `allowed-tools` to restrict tools.

## What you'll observe

- Frontmatter parses into settings; `$ARGUMENTS` and `$1` fill in at invoke time.
- `allowed-tools` blocks a tool the skill is not permitted to use.
- A `context: fork` skill keeps its verbose work in a subagent and returns only a short summary.

## How to run

Run top to bottom. Building the files and the pure-Python mechanics run anywhere. The live cell
invokes the project command through Claude, so paste a real key into **Setup 2/3** and re-run from
the top; otherwise it skips. **Node.js 18+** must be installed for the Agent SDK.

## 0. Setup

**This cell:** installs the packages. `pyyaml` parses the YAML frontmatter; the Agent SDK drives
the live invocation and needs Node.js 18+.

In [ ]:
# ===== SETUP 1/3 - install the packages =====
%pip install -q claude-agent-sdk anthropic python-dotenv pyyaml

**This cell:** imports, the model, the `RUN_LIVE` switch, and `run_async()` for the live cell.

In [ ]:
# ===== SETUP 2/3 - imports, the model, the switch, and an async runner =====
import os                                       # filesystem paths for the sandbox project
import re                                       # simple text substitution
import sys                                       # detect Windows (special event loop)
import yaml                                     # parse the YAML frontmatter
import asyncio                                  # the Agent SDK is async; we drive it ourselves
import threading                                # run that async loop in a side thread (notebook-safe)

try:                                            # load a .env file if present
    from dotenv import load_dotenv              #   import the loader
    load_dotenv()                               #   read .env into environment variables
except Exception:                               # not installed? that is fine
    pass                                        #   set the key another way

MODEL = "claude-sonnet-4-6"                      # the Sonnet model the live cell will use

os.environ.setdefault("ANTHROPIC_API_KEY", "sk-ant-...")     # placeholder unless you set a real key
_key = os.environ["ANTHROPIC_API_KEY"]           # read whatever key is set
RUN_LIVE = _key.startswith("sk-ant-") and _key != "sk-ant-..."   # True only for a real key

def run_async(make_coro):                        # run any async Agent SDK call, notebook-safe
    box = {}                                     #   carries the result/error out of the thread
    def worker():                                #   runs in its own thread
        loop = asyncio.ProactorEventLoop() if sys.platform == "win32" else asyncio.new_event_loop()
        asyncio.set_event_loop(loop)             #     make it this thread's loop
        try:    box["value"] = loop.run_until_complete(make_coro())   # run to completion
        except Exception as e: box["error"] = e  #     capture any error
        finally: loop.close()                    #     always close the loop
    t = threading.Thread(target=worker); t.start(); t.join()   # run it and wait
    if "error" in box: raise box["error"]        #   surface any error here
    return box.get("value")                      #   hand back the result

print("live model calls:", "ON" if RUN_LIVE else "OFF (using a placeholder key)")

**This cell:** writes a small **sandbox project**: a `/review` command, a `refund-audit` skill,
and one ShopDesk module for them to act on. We use a sandbox folder so nothing touches your real
`~/.claude`. The command and skill are plain text files, exactly what you would commit to a repo.

In [ ]:
# ===== SETUP 3/3 - create the sandbox project (command + skill + a module) =====
import textwrap                                    # keeps the embedded file bodies readable
PROJECT = os.path.join(os.getcwd(), "shopdesk_project")   # the sandbox project root

FILES = {
    ".claude/commands/review.md": textwrap.dedent("""\
        ---
        description: Review a ShopDesk module against our checklist
        argument-hint: [file-path]
        allowed-tools: Read, Grep
        ---
        Review $ARGUMENTS against the ShopDesk checklist:
        1. Every refund path checks the 30-day window (REFUND_WINDOW_DAYS).
        2. No secret keys are hardcoded.
        3. Public functions have a docstring.
        Report findings as a short bulleted list. Do not edit anything.
        """),
    ".claude/skills/refund-audit/SKILL.md": textwrap.dedent("""\
        ---
        name: refund-audit
        description: Audit refund logic for window checks and hardcoded secrets. Use before shipping refund changes.
        context: fork
        allowed-tools: Read, Grep
        argument-hint: [directory]
        ---
        Audit the refund logic under $ARGUMENTS.
        1. Grep for refund functions.
        2. Confirm each checks REFUND_WINDOW_DAYS.
        3. Flag any hardcoded secret (for example sk-ant-... or ghp_...).
        Return only a short summary of findings.
        """),
    "shopdesk/refunds.py": textwrap.dedent("""\
        REFUND_WINDOW_DAYS = 30

        def is_refundable(order):
            return order["delivered_days_ago"] <= REFUND_WINDOW_DAYS

        def process_refund(order):
            return "refunded" if is_refundable(order) else "refused"
        """),
}
for rel, content in FILES.items():                 # write every file
    path = os.path.join(PROJECT, rel)
    os.makedirs(os.path.dirname(path), exist_ok=True)
    with open(path, "w") as f:
        f.write(content)
print("created project at", PROJECT, "with", len(FILES), "files")

### Commands and skills, side by side

- A **slash command** is a Markdown file in `.claude/commands/` (project, shared via Git) or
  `~/.claude/commands/` (personal). The filename is the command name: `review.md` becomes `/review`.
- A **skill** is a folder in `.claude/skills/<name>/` with a `SKILL.md`. It does everything a command
  does, plus richer frontmatter and supporting files.
- Both use YAML frontmatter: `description`, `allowed-tools`, `argument-hint`, `model`. Skills add
  `context: fork` to run in an isolated subagent.
- Arguments: `$ARGUMENTS` is the whole trailing string; `$1`, `$2` are positional.

**Worth knowing:** since Claude Code v2.1.101 (April 2026) commands and skills were **merged**, so
both create a `/name` you can invoke, and if a command and a skill share a name the skill wins. Your
existing `.claude/commands/` files still work, so this lab builds both.

---

### Lab objective - package, parameterize, restrict, isolate

**What you build:** a `/review` command and a `refund-audit` skill, then the mechanics that fill in
arguments, enforce `allowed-tools`, and isolate a `context: fork` run.

**Why it helps you build real solutions:** turning a checklist into a versioned command means every
teammate runs the same steps, and `allowed-tools` plus `context: fork` keep a skill safe and its noise
out of your main conversation.

**How you'll see it:** the parsed frontmatter drives behavior, a disallowed tool is blocked, and a
forked skill returns only a summary.

**This cell:** a small parser that splits a command or skill file into its **frontmatter** (the
YAML between the `---` fences) and its **body** (the prompt template). This is exactly what Claude Code
does when it loads the file.

In [ ]:
# ===== parse a command/skill file into (frontmatter, body) =====
def parse_config(rel):                             # rel path under PROJECT -> (dict, body_text)
    text = open(os.path.join(PROJECT, rel)).read()
    m = re.match(r"^---\n(.*?)\n---\n(.*)$", text, re.DOTALL)   # match the fenced frontmatter
    if not m:
        return {}, text                             #   no frontmatter -> all body
    front = yaml.safe_load(m.group(1)) or {}        #   parse the YAML block
    return front, m.group(2)                        #   return settings and the prompt body

front, body = parse_config(".claude/commands/review.md")   # load the /review command
print("frontmatter:", front)
print("body starts:", body.splitlines()[0])

**This cell:** argument substitution. When you type `/review shopdesk/refunds.py`, Claude Code
replaces `$ARGUMENTS` with that text (and `$1`, `$2` with positional words) before running the prompt.
We do the same here so you can see the final prompt.

In [ ]:
# ===== fill $ARGUMENTS and $1, $2 in the body =====
def apply_args(body, argstr):                      # body + "a b c" -> body with args substituted
    parts = argstr.split()                          #   positional words
    filled = body.replace("$ARGUMENTS", argstr)     #   the whole trailing string
    for i, word in enumerate(parts, 1):             #   $1, $2, ... one-indexed
        filled = filled.replace(f"${i}", word)
    return filled

final_prompt = apply_args(body, "shopdesk/refunds.py")   # invoke: /review shopdesk/refunds.py
print(final_prompt.splitlines()[0])                 # the first line now names the file

**This cell:** the `allowed-tools` restriction. The `/review` command lists only `Read` and
`Grep`, so a review must not edit. We parse that list and check tools against it: `Read` is allowed,
`Write` is blocked. This is how a command or skill is kept read-only.

In [ ]:
# ===== enforce allowed-tools =====
def allowed_set(front):                            # frontmatter -> set of permitted base tool names
    spec = front.get("allowed-tools", "")           #   e.g. "Read, Grep" or "Bash(git add:*)"
    names = [t.strip().split("(")[0] for t in spec.split(",") if t.strip()]   # base name before "("
    return set(names)

allowed = allowed_set(front)                        # tools the /review command may use
print("allowed:", allowed)
for tool in ["Read", "Grep", "Write"]:              # check three tools against the list
    print(f"  {tool:5} -> {'allowed' if tool in allowed else 'BLOCKED'}")

**This cell:** what `context: fork` buys you. A forked skill runs in an isolated subagent: it may
read and search many files (verbose work), but only a short **summary** returns to your main
conversation. We simulate the token accounting so the saving is visible.

In [ ]:
# ===== simulate context: fork isolation =====
skill_front, skill_body = parse_config(".claude/skills/refund-audit/SKILL.md")   # load the skill
is_forked = skill_front.get("context") == "fork"    # does it isolate execution?

fork_verbose = "reads refunds.py, greps 3 helpers, inspects 40 lines of context " * 20   # noisy work
returned_summary = "1 refund path found; window check present; no hardcoded secrets."     # what comes back

print("context: fork ->", is_forked)
print("tokens produced inside the fork (approx):", len(fork_verbose) // 4)
print("tokens returned to your main context     :", len(returned_summary) // 4)
print("the verbose work never enters your main conversation")

**This cell:** the live run. We point the Agent SDK at the sandbox project with `cwd=PROJECT`
and `setting_sources=["project"]` so it loads `.claude/commands/`, then invoke `/review`. With only
`Read`, `Grep`, and `Glob` allowed, the review stays read-only.

In [ ]:
# ===== live: invoke the project slash command =====
from claude_agent_sdk import query, ClaudeAgentOptions, AssistantMessage, TextBlock, ToolUseBlock

CMD_OPTS = ClaudeAgentOptions(                     # load project commands/skills from the sandbox
    model=MODEL, cwd=PROJECT,
    setting_sources=["project"],                    # this is what loads .claude/commands and skills
    allowed_tools=["Read", "Grep", "Glob"])         # read-only, matching the command's intent

async def run_cmd(prompt):                          # stream the run and show tool calls + text
    async for m in query(prompt=prompt, options=CMD_OPTS):
        if isinstance(m, AssistantMessage):
            for b in m.content:
                if isinstance(b, ToolUseBlock): print("  ->", b.name)
                elif isinstance(b, TextBlock) and b.text.strip(): print("  ", b.text.strip()[:160])

if RUN_LIVE:                                        # needs a real key (and Node.js 18+)
    run_async(lambda: run_cmd("/review shopdesk/refunds.py"))
else:
    print("[skipped] expected: Claude loads .claude/commands/review.md and reviews the file read-only.")
    print("          (setting_sources=['project'] is what makes the /review command available).")

**In the real Claude Code CLI** (this is reference, not run here):

```text
# from the project root, the files you created are picked up automatically
/review shopdesk/refunds.py     # run the project command with an argument
/refund-audit shopdesk          # run the skill (isolated via context: fork)
/help                           # list available commands and skills
```

Project files in `.claude/commands/` and `.claude/skills/` are shared through Git; personal ones live
under `~/.claude/` and are not shared with your team.

| anti-pattern | what to do instead |
|---|---|
| paste the same checklist prompt every time | put it in `.claude/commands/` as `/review` |
| give an audit skill full tool access | set `allowed-tools` to the minimum (Read, Grep) |
| let a verbose skill flood your context | add `context: fork` to isolate it |
| keep a shared workflow in `~/.claude/` | use project scope so teammates get it on clone |

**Lesson:** a **command** is a shared prompt template; a **skill** is the same idea with richer
frontmatter and its own folder. Frontmatter is where the control lives: `allowed-tools` restricts what
the workflow can do, and `context: fork` runs it in an isolated subagent so only a summary returns.
Package the workflows your team repeats, and keep each one least-privileged.

---

## Recap - commands and skills

| Piece | Where | What it does |
|---|---|---|
| slash command | `.claude/commands/*.md` | shared prompt template, `$ARGUMENTS` |
| skill | `.claude/skills/<name>/SKILL.md` | command plus frontmatter and files |
| `allowed-tools` | frontmatter | restrict tools (least privilege) |
| `context: fork` | skill frontmatter | isolate execution, return a summary |

One principle to carry forward: **codify the repeated workflow, then restrict and isolate it.** To run
live, paste a real key into **Setup 2/3** and re-run from the top. Then try it: add a `$1`/`$2`
positional command and invoke it with two arguments. Next lab: plan mode, direct execution, and the
Explore subagent.